# Pan-cancer cohort: immunotherapy versus chemotherapy

This notebook queries cancer diagnoses, study-drug exposures, and death records inside the All of Us Researcher Workbench, then exports the selected patients' longitudinal events for downstream treatment-effect preparation. Run it with authorised data access and a configured `WORKSPACE_CDR`. The generated queries target Registered Tier v8.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (path / "src" / "model.py").is_file() and (path / "runnables").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from the repository or one of its subdirectories.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.project_paths import SHARED_DATA_DIR, COHORT_DATA_DIR

import pandas
import os
import json
from pathlib import Path

def _load_all_of_us_config():
    config_path = PROJECT_ROOT / ".all_of_us_config.json"
    if not config_path.exists():
        return {}
    return json.loads(config_path.read_text())

def _ensure_workspace_cdr(config):
    if "WORKSPACE_CDR" in os.environ:
        return

    workspace_cdr = config.get("WORKSPACE_CDR")
    cdr_project = config.get("CDR_VERSION_CLOUD_PROJECT")
    cdr_dataset = config.get("CDR_VERSION_BIGQUERY_DATASET")
    if workspace_cdr:
        os.environ["WORKSPACE_CDR"] = workspace_cdr
    elif cdr_project and cdr_dataset:
        os.environ["WORKSPACE_CDR"] = f"{cdr_project}.{cdr_dataset}"
    else:
        raise KeyError("WORKSPACE_CDR is not set and the All of Us config is missing CDR project/dataset values")

all_of_us_config = _load_all_of_us_config()
_ensure_workspace_cdr(all_of_us_config)
BIGQUERY_PROJECT = (
    os.environ.get("GOOGLE_CLOUD_PROJECT")
    or os.environ.get("GOOGLE_PROJECT")
    or all_of_us_config.get("BILLING_CLOUD_PROJECT")
)
BIGQUERY_LOCATION = os.environ.get("BIGQUERY_LOCATION") or all_of_us_config.get("BIGQUERY_LOCATION") or "us-central1"
USE_BQSTORAGE_API = os.environ.get("BIGQUERY_STORAGE_API_ENABLED", "1") != "0"

import pandas as pd
from IPython.display import display

OUTPUT_DIR = COHORT_DATA_DIR / "PAN_CANCER"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Coverage filters below apply to diagnostic summaries, not the exported population.
BASELINE_DAYS_MIN = 0
FOLLOWUP_DAYS_MIN = 0
MORTALITY_WINDOWS_DAYS = {"Death within 24 months": 730, "Death within 36 months": 1095}

CANCER_CODES = ['C15','C16','C17','C18','C19','C20','C21','C22','C23','C24','C25','C26',
               'C30','C31','C32','C33','C34','C35','C36','C37','C38','C39',
               'C43',
               'C50','C51','C52','C53','C54','C55','C56','C57','C58',
               'C60','C61','C62','C63','C64','C65','C66','C67','C68',]

## Query cancer diagnoses

In [ ]:
import pandas_gbq
import pandas
import os

# This query represents dataset "Neoplasms" for domain "condition" and was generated for All of Us Registered Tier Dataset v8
dataset_33674864_condition_sql = """
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,

        c_occurrence.condition_start_datetime,
        c_occurrence.condition_source_value,
        c_source_concept.concept_code as source_concept_code,
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.condition_occurrence` c_occurrence 
        WHERE
            (
                condition_source_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                    WHERE
                        concept_id IN (1326484, 1326485, 1326486, 1326488, 1326489, 1326490, 1567470, 1567474, 1567475, 1567477, 1567479, 1567485, 1567486, 1567490, 1567491, 1567502, 1567515, 1567520, 1567537, 1567563, 1567564, 1567566, 1567567, 1567568, 1567574, 1567577, 1567580, 1567581, 1567584, 1567596, 1567597, 1567603, 1567608, 1567619, 1567620, 1567626, 1567631, 1567632, 1567633, 1567640, 1567641, 1567642, 1567652, 1567653, 1567654, 1567656, 1567660, 1567661, 1567672, 1567675, 1567676, 1567681, 1567682, 1567690, 1567691, 1567694, 1567707, 1567711, 1567715, 1567722, 1567729, 1567735, 1567736, 1567738, 1567740, 1567743, 1567745, 1567747, 1567751, 1567756, 1567761, 1567765, 1567767, 1567771, 1567777, 1567783, 1567784, 1567799, 1567804, 1567805, 1567807, 1567831, 1595483, 1595484, 1595486, 1595487, 1595491, 1595492, 1595493, 1595494, 1595495, 1595496, 1595497, 1595498, 1595499, 1595500, 1595501, 1595502, 1595504, 1595506, 1595509, 1595510, 1595511, 1595512, 1595513, 1595514, 1595516,
 1595517, 1595518, 1595519, 1595520, 1595521, 1595522, 1595523, 1595524, 1595525, 1595526, 1595527, 1595528, 1595529, 1595530, 35206047, 35206048, 35206049, 35206050, 35206051, 35206053, 35206054, 35206055, 35206056, 35206057, 35206058, 35206059, 35206060, 35206061, 35206062, 35206063, 35206064, 35206065, 35206066, 35206067, 35206068, 35206069, 35206070, 35206071, 35206072, 35206073, 35206074, 35206075, 35206076, 35206077, 35206078, 35206079, 35206080, 35206081, 35206082, 35206083, 35206084, 35206085, 35206086, 35206087, 35206088, 35206089, 35206090, 35206091, 35206092, 35206093, 35206094, 35206095, 35206096, 35206097, 35206098, 35206099, 35206100, 35206101, 35206102, 35206103, 35206104, 35206105, 35206106, 35206107, 35206109, 35206110, 35206111, 35206112, 35206113, 35206114, 35206115, 35206116, 35206117, 35206118, 35206119, 35206120, 35206121, 35206122, 35206123, 35206124, 35206125, 35206126, 35206127, 35206128, 35206129, 35206130, 35206131, 35206132, 35206133, 35206134, 35206135,
 35206136, 35206137, 35206138, 35206139, 35206140, 35206141, 35206142, 35206143, 35206144, 35206145, 35206146, 35206147, 35206148, 35206149, 35206150, 35206151, 35206152, 35206153, 35206154, 35206155, 35206156, 35206157, 35206158, 35206159, 35206160, 35206161, 35206162, 35206163, 35206164, 35206165, 35206166, 35206167, 35206168, 35206169, 35206170, 35206171, 35206172, 35206173, 35206174, 35206175, 35206176, 35206177, 35206178, 35206179, 35206180, 35206181, 35206182, 35206183, 35206184, 35206185, 35206186, 35206187, 35206188, 35206189, 35206190, 35206191, 35206192, 35206193, 35206194, 35206195, 35206196, 35206197, 35206198, 35206199, 35206200, 35206201, 35206202, 35206203, 35206208, 35206209, 35206210, 35206211, 35206212, 35206213, 35206214, 35206215, 35206216, 35206217, 35206218, 35206219, 35206220, 35206221, 35206222, 35206223, 35206224, 35206225, 35206226, 35206227, 35206228, 35206229, 35206230, 35206231, 35206232, 35206233, 35206234, 35206235, 35206236, 35206237, 35206238, 35206239,
 35206240, 35206241, 35206242, 35206243, 35206244, 35206245, 35206246, 35206247, 35206248, 35206249, 35206250, 35206251, 35206252, 35206253, 35206255, 35206256, 35206257, 35206258, 35206259, 35206260, 35206261, 35206262, 35206263, 35206264, 35206265, 35206266, 35206267, 35206268, 35206269, 35206270, 35206274, 35206275, 35206276, 35206277, 35206278, 35206279, 35206280, 35206281, 35206282, 35206283, 35206284, 35206285, 35206286, 35206287, 35206288, 35206289, 35206290, 35206291, 35206292, 35206293, 35206294, 35206295, 35206296, 35206297, 35206298, 35206299, 35206300, 35206301, 35206302, 35206303, 35206304, 35206305, 35206306, 35206307, 35206308, 35206309, 35206310, 35206311, 35206312, 35206313, 35206314, 35206315, 35206316, 35206318, 35206319, 35206320, 35206321, 35206322, 35206323, 35206324, 35206325, 35206326, 35206327, 35206328, 35206329, 35206330, 35206331, 35206332, 35206333, 35206334, 35206425, 35206427, 35206428, 35206430, 35206451, 35206453, 35206456, 35206457, 35206458, 35206459,
 35206461, 35206462, 35206463, 35206464, 35206465, 35206466, 35206467, 35206468, 35206469, 35206470, 35206471, 35206472, 35206473, 35206474, 35206475, 35206476, 35206477, 35206478, 35206485, 35206486, 35206487, 35206488, 35206489, 35206490, 35206491, 35206492, 35206493, 35206494, 35206495, 35206497, 35206498, 35206499, 35206500, 35206501, 35206502, 35206503, 35206504, 35206505, 35206506, 35206507, 35206508, 35206509, 35206510, 35206511, 35206512, 35206513, 35206514, 35206515, 35206516, 35206517, 35206518, 35206519, 35206520, 35206521, 35206522, 35206523, 35206524, 35206525, 35206526, 35206527, 35206528, 35206529, 35206530, 35206531, 35206532, 35206533, 35206535, 35206536, 35206537, 35206538, 35206539, 35206540, 35206541, 35206542, 35206543, 35206544, 35206545, 35206546, 35206548, 35206549, 35206550, 35206551, 35206552, 35206553, 35206554, 35206555, 35206556, 35206557, 35206558, 35206559, 35206560, 35206561, 35206562, 35206563, 35206564, 35206565, 35206566, 35206567, 35206568, 35206569,
 35206576, 35206577, 35206578, 35206579, 35206580, 35206581, 35206582, 35206583, 35206584, 35206585, 35206586, 35206587, 35206588, 35206589, 35206590, 35206591, 35206593, 35206594, 35206595, 35206597, 35206598, 35206599, 35206600, 35206601, 35206602, 35206603, 35206604, 35206605, 35206606, 35206607, 35206608, 35206609, 35206610, 35206611, 35206612, 35206613, 35206614, 35206615, 35206617, 35206618, 35206619, 35206620, 35206621, 35206622, 35206623, 35206624, 35206625, 35206626, 35206628, 35206629, 35206630, 35206631, 35206632, 35206633, 35206634, 35206635, 35206636, 35206637, 35206639, 35206640, 35206642, 35206643, 35206644, 35206646, 35206647, 35206648, 35206649, 35206650, 35206651, 35206652, 35206653, 35206654, 35206656, 35206657, 35206658, 35206659, 35206660, 35206661, 35206662, 35206663, 35206665, 35206666, 35206667, 35206668, 35206670, 35206675, 35206676, 35206677, 35206678, 35206679, 35206681, 35206682, 35206683, 35206684, 35206685, 35206686, 35206687, 35206689, 35206690, 37200003,
 37200004, 37200005, 37200006, 37200007, 37200008, 37200009, 37200010, 37200012, 37200013, 37200014, 37200015, 45532899, 45532904, 45532907, 45532908, 45532909, 45532910, 45532911, 45532912, 45532915, 45532916, 45532919, 45532920, 45532921, 45532923, 45532926, 45532927, 45532928, 45532932, 45532937, 45532939, 45532940, 45532941, 45532942, 45532943, 45532944, 45532945, 45532947, 45532948, 45532950, 45532951, 45532952, 45532953, 45532955, 45532958, 45532959, 45532960, 45532961, 45532963, 45532964, 45532965, 45532968, 45532969, 45532972, 45532973, 45532974, 45532976, 45532977, 45532978, 45532980, 45532981, 45537790, 45537808, 45537809, 45537810, 45537811, 45537812, 45537813, 45537820, 45537821, 45537824, 45537828, 45537832, 45537833, 45537834, 45537840, 45537842, 45537843, 45537844, 45537846, 45537847, 45537848, 45537849, 45537850, 45537851, 45537852, 45537853, 45537854, 45537856, 45537857, 45537858, 45537859, 45537860, 45537862, 45537863, 45537865, 45537866, 45537867, 45537868, 45537869,
 45537870, 45537872, 45537874, 45537875, 45537877, 45537878, 45537883, 45537884, 45537886, 45537887, 45537888, 45537893, 45537897, 45537898, 45537900, 45537901, 45537902, 45537904, 45537907, 45537908, 45537913, 45542591, 45542593, 45542594, 45542595, 45542598, 45542599, 45542600, 45542601, 45542602, 45542603, 45542604, 45542605, 45542606, 45542607, 45542608, 45542609, 45542613, 45542615, 45542617, 45542619, 45542622, 45542623, 45542628, 45542629, 45542630, 45542631, 45542632, 45542633, 45542634, 45542635, 45542636, 45542637, 45542639, 45542640, 45542641, 45542643, 45542644, 45542645, 45542647, 45542648, 45542650, 45542652, 45542653, 45542655, 45542657, 45542658, 45542660, 45542662, 45542664, 45542665, 45542668, 45542669, 45542672, 45542674, 45542676, 45542678, 45542679, 45542685, 45542686, 45542689, 45542690, 45542694, 45542699, 45542701, 45542704, 45547486, 45547487, 45547488, 45547489, 45547490, 45547492, 45547495, 45547496, 45547497, 45547498, 45547499, 45547502, 45547503, 45547504,
 45547510, 45547511, 45547513, 45547514, 45547515, 45547518, 45547521, 45547525, 45547526, 45547527, 45547533, 45547534, 45547535, 45547536, 45547537, 45547538, 45547539, 45547540, 45547541, 45547543, 45547544, 45547545, 45547548, 45547549, 45547550, 45547551, 45547552, 45547553, 45547554, 45547555, 45547557, 45547562, 45547563, 45547566, 45547567, 45547568, 45547569, 45547570, 45547571, 45547573, 45547579, 45547580, 45547583, 45547584, 45547587, 45547590, 45547592, 45547595, 45547597, 45547598, 45547599, 45547600, 45547601, 45547603, 45552252, 45552253, 45552254, 45552255, 45552256, 45552259, 45552260, 45552262, 45552267, 45552273, 45552274, 45552276, 45552278, 45552281, 45552283, 45552285, 45552286, 45552287, 45552288, 45552289, 45552290, 45552291, 45552294, 45552295, 45552296, 45552297, 45552298, 45552300, 45552301, 45552302, 45552303, 45552304, 45552305, 45552306, 45552308, 45552311, 45552313, 45552314, 45552316, 45552317, 45552318, 45552320, 45552321, 45552330, 45552332, 45552333,
 45552336, 45552337, 45552338, 45552342, 45552343, 45552344, 45552346, 45552354, 45552355, 45556973, 45556978, 45556979, 45556981, 45556982, 45556990, 45556992, 45556993, 45556994, 45556995, 45556996, 45557001, 45557002, 45557003, 45557009, 45557010, 45557011, 45557015, 45557016, 45557018, 45557019, 45557020, 45557021, 45557022, 45557023, 45557024, 45557025, 45557026, 45557027, 45557028, 45557030, 45557031, 45557032, 45557033, 45557034, 45557035, 45557036, 45557038, 45557039, 45557040, 45557043, 45557045, 45557046, 45557052, 45557053, 45557060, 45557062, 45557063, 45557066, 45557070, 45557071, 45557072, 45557077, 45557078, 45557079, 45557080, 45557081, 45557082, 45561783, 45561794, 45561795, 45561796, 45561797, 45561798, 45561799, 45561800, 45561801, 45561802, 45561803, 45561804, 45561807, 45561810, 45561812, 45561813, 45561815, 45561817, 45561818, 45561819, 45561820, 45561822, 45561823, 45561824, 45561825, 45561826, 45561833, 45561834, 45561835, 45561836, 45561837, 45561838, 45561839,
 45561840, 45561841, 45561842, 45561843, 45561844, 45561845, 45561846, 45561847, 45561848, 45561849, 45561850, 45561851, 45561852, 45561853, 45561854, 45561855, 45561856, 45561859, 45561860, 45561861, 45561863, 45561864, 45561865, 45561866, 45561868, 45561871, 45561872, 45561873, 45561875, 45561876, 45561877, 45561879, 45561885, 45561890, 45561893, 45561900, 45561904, 45561908, 45561910, 45561911, 45561914, 45561915, 45561916, 45566582, 45566584, 45566586, 45566588, 45566590, 45566592, 45566593, 45566594, 45566595, 45566596, 45566597, 45566599, 45566600, 45566602, 45566603, 45566607, 45566614, 45566615, 45566616, 45566618, 45566619, 45566621, 45566623, 45566625, 45566626, 45566627, 45566628, 45566629, 45566630, 45566632, 45566633, 45566634, 45566635, 45566636, 45566637, 45566638, 45566639, 45566641, 45566642, 45566643, 45566644, 45566645, 45566647, 45566649, 45566651, 45566655, 45566658, 45566659, 45566661, 45566663, 45566665, 45566668, 45566669, 45566672, 45566674, 45566675, 45566676,
 45566679, 45566681, 45566684, 45566685, 45566686, 45566687, 45566688, 45566692, 45566695, 45566696, 45571493, 45571494, 45571497, 45571499, 45571502, 45571503, 45571504, 45571505, 45571506, 45571507, 45571509, 45571513, 45571517, 45571519, 45571520, 45571521, 45571522, 45571523, 45571525, 45571530, 45571531, 45571536, 45571537, 45571541, 45571542, 45571543, 45571546, 45571547, 45571548, 45571550, 45571551, 45571552, 45571554, 45571557, 45571559, 45571561, 45571562, 45571563, 45571564, 45571565, 45571567, 45571568, 45571569, 45571573, 45571574, 45571577, 45571578, 45571579, 45571580, 45571583, 45571584, 45571585, 45571587, 45571588, 45571593, 45571594, 45571600, 45571604, 45571607, 45571608, 45571609, 45571615, 45571616, 45571617, 45571618, 45571622, 45571625, 45571626, 45571627, 45576310, 45576313, 45576314, 45576316, 45576319, 45576321, 45576322, 45576323, 45576325, 45576327, 45576330, 45576334, 45576336, 45576340, 45576341, 45576342, 45576343, 45576345, 45576346, 45576347, 45576348,
 45576349, 45576350, 45576351, 45576352, 45576354, 45576355, 45576356, 45576357, 45576358, 45576360, 45576361, 45576362, 45576363, 45576364, 45576365, 45576366, 45576368, 45576369, 45576371, 45576375, 45576378, 45576380, 45576381, 45576383, 45576384, 45576385, 45576387, 45576388, 45576393, 45576394, 45576399, 45576402, 45581204, 45581205, 45581206, 45581208, 45581209, 45581210, 45581211, 45581213, 45581216, 45581222, 45581223, 45581224, 45581225, 45581228, 45581229, 45581230, 45581231, 45581233, 45581234, 45581235, 45581236, 45581237, 45581238, 45581239, 45581240, 45581241, 45581243, 45581245, 45581246, 45581247, 45581248, 45581251, 45581253, 45581255, 45581257, 45581260, 45581261, 45581266, 45581267, 45581268, 45581269, 45581270, 45581272, 45581273, 45581274, 45581276, 45581277, 45581286, 45581287, 45581288, 45581290, 45581292, 45581293, 45581299, 45581300, 45581301, 45581309, 45581310, 45581312, 45581313, 45581314, 45581315, 45581319, 45585987, 45585992, 45585993, 45585994, 45585995,
 45585996, 45585998, 45586000, 45586001, 45586002, 45586004, 45586005, 45586006, 45586009, 45586010, 45586011, 45586012, 45586013, 45586014, 45586017, 45586018, 45586019, 45586021, 45586022, 45586023, 45586026, 45586027, 45586029, 45586031, 45586033, 45586034, 45586035, 45586036, 45586037, 45586038, 45586039, 45586040, 45586041, 45586042, 45586043, 45586045, 45586046, 45586047, 45586048, 45586049, 45586050, 45586052, 45586053, 45586054, 45586055, 45586057, 45586058, 45586060, 45586063, 45586064, 45586065, 45586066, 45586067, 45586068, 45586069, 45586070, 45586072, 45586077, 45586079, 45586081, 45586091, 45586092, 45586093, 45586095, 45586097, 45586098, 45586100, 45586101, 45586103, 45586105, 45586110, 45590884, 45590885, 45590888, 45590889, 45590890, 45590891, 45590892, 45590893, 45590894, 45590895, 45590896, 45590897, 45590898, 45590899, 45590901, 45590903, 45590904, 45590905, 45590906, 45590908, 45590909, 45590910, 45590911, 45590913, 45590914, 45590916, 45590917, 45590927, 45590928,
 45590929, 45590931, 45590932, 45590933, 45590938, 45590940, 45590941, 45590942, 45590943, 45590944, 45590946, 45590948, 45590949, 45590951, 45590952, 45590954, 45590955, 45590956, 45590957, 45590959, 45590960, 45590963, 45590964, 45590966, 45590967, 45590968, 45590970, 45590972, 45590974, 45590977, 45590979, 45590980, 45590981, 45590989, 45590990, 45590995, 45590998, 45590999, 45591000, 45591004, 45591005, 45591007, 45591008, 45591009, 45595653, 45595654, 45595655, 45595657, 45595659, 45595661, 45595663, 45595664, 45595665, 45595666, 45595667, 45595668, 45595672, 45595673, 45595677, 45595678, 45595679, 45595680, 45595681, 45595684, 45595689, 45595690, 45595691, 45595692, 45595693, 45595694, 45595700, 45595701, 45595702, 45595703, 45595704, 45595705, 45595706, 45595707, 45595709, 45595710, 45595712, 45595713, 45595716, 45595717, 45595718, 45595719, 45595720, 45595721, 45595722, 45595723, 45595724, 45595726, 45595728, 45595729, 45595730, 45595731, 45595732, 45595735, 45595737, 45595738,
 45595743, 45595744, 45595745, 45595746, 45595747, 45595748, 45595749, 45595750, 45595753, 45595755, 45595756, 45595757, 45595758, 45595763, 45600488, 45600489, 45600490, 45600491, 45600494, 45600495, 45600497, 45600498, 45600499, 45600500, 45600501, 45600504, 45600505, 45600506, 45600507, 45600508, 45600510, 45600511, 45600515, 45600517, 45600522, 45600523, 45600524, 45600525, 45600526, 45600527, 45600528, 45600530, 45600531, 45600532, 45600533, 45600534, 45600535, 45600536, 45600537, 45600538, 45600539, 45600540, 45600541, 45600542, 45600543, 45600544, 45600545, 45600547, 45600548, 45600550, 45600551, 45600553, 45600554, 45600555, 45600556, 45600558, 45600559, 45600560, 45600563, 45600567, 45600569, 45600570, 45600574, 45600575, 45600576, 45600577, 45600581, 45600583, 45600584, 45600588, 45600590, 45600593, 45600594, 45600600, 45600601, 45600602, 45605267, 45605268, 45605269, 45605270, 45605271, 45605272, 45605275, 45605276, 45605277, 45605279, 45605284, 45605285, 45605286, 45605288,
 45605289, 45605290, 45605291, 45605292, 45605294, 45605298, 45605299, 45605301, 45605302, 45605304, 45605305, 45605306, 45605308, 45605309, 45605310, 45605311, 45605313, 45605314, 45605315, 45605316, 45605317, 45605318, 45605320, 45605321, 45605323, 45605324, 45605325, 45605326, 45605327, 45605329, 45605330, 45605331, 45605334, 45605335, 45605336, 45605339, 45605340, 45605341, 45605346, 45605347, 45605349, 45605350, 45605351, 45605352, 45605354, 45605355, 45605356, 45605360, 45605361, 45605362, 45605367, 45605371, 45605372, 766339, 766340, 766341, 920115, 920116, 920117, 920118, 920119, 920120, 920121, 920122, 920123, 920124, 920125, 920126, 920127)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 0 
                    AND is_selectable = 1)
            )  
            AND (
                c_occurrence.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p)   
                    AND cb_search_person.person_id IN (SELECT
                        person_id  
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_ehr_data = 1 ) )
            )) c_occurrence 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_standard_concept 
            ON c_occurrence.condition_concept_id = c_standard_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_type 
            ON c_occurrence.condition_type_concept_id = c_type.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
            ON c_occurrence.visit_occurrence_id = v.visit_occurrence_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` visit 
            ON v.visit_concept_id = visit.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_source_concept 
            ON c_occurrence.condition_source_concept_id = c_source_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_status 
            ON c_occurrence.condition_status_concept_id = c_status.concept_id"""

dataset_33674864_condition_df = pandas_gbq.read_gbq(
    dataset_33674864_condition_sql,
    dialect="standard",
    project_id=BIGQUERY_PROJECT,
    location=BIGQUERY_LOCATION,
    use_bqstorage_api=USE_BQSTORAGE_API,
    progress_bar_type="tqdm_notebook")

## Query chemotherapy exposures

In [ ]:
import pandas_gbq
# chemotherapy
import pandas
import os

# This query represents dataset "Chemotherapy" for domain "drug" and was generated for All of Us Registered Tier Dataset v8
dataset_08473385_drug_sql = """
    SELECT
        d_exposure.person_id,
        d_exposure.drug_concept_id,
        d_standard_concept.concept_name as standard_concept_name,
        d_standard_concept.concept_code as standard_concept_code,
        d_standard_concept.vocabulary_id as standard_vocabulary,
        d_exposure.drug_exposure_start_datetime,
        d_type.concept_name as drug_type_concept_name,
        d_visit.concept_name as visit_occurrence_concept_name,
        d_exposure.drug_source_concept_id,
        d_source_concept.concept_name as source_concept_name,
        d_source_concept.concept_code as source_concept_code,
        d_source_concept.vocabulary_id as source_vocabulary,
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.drug_exposure` d_exposure 
        WHERE
            (
                drug_concept_id IN (SELECT
                    DISTINCT ca.descendant_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                JOIN
                    (SELECT
                        DISTINCT c.concept_id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id             
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                        WHERE
                            concept_id IN (21601388, 21601421, 21601445, 21603728, 21603747)             
                            AND full_text LIKE '%_rank1]%'       ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) b 
                        ON (ca.ancestor_id = b.concept_id)))  
                    AND (d_exposure.PERSON_ID IN (SELECT
                        distinct person_id  
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                   ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_ehr_data = 1 ) )
            )) d_exposure 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_standard_concept 
            ON d_exposure.drug_concept_id = d_standard_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_type 
            ON d_exposure.drug_type_concept_id = d_type.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_route 
            ON d_exposure.route_concept_id = d_route.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
            ON d_exposure.visit_occurrence_id = v.visit_occurrence_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_visit 
            ON v.visit_concept_id = d_visit.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_source_concept 
            ON d_exposure.drug_source_concept_id = d_source_concept.concept_id"""

dataset_08473385_drug_df = pandas_gbq.read_gbq(
    dataset_08473385_drug_sql,
    dialect="standard",
    project_id=BIGQUERY_PROJECT,
    location=BIGQUERY_LOCATION,
    use_bqstorage_api=USE_BQSTORAGE_API,
    progress_bar_type="tqdm_notebook")

## Query immunotherapy exposures

In [ ]:
import pandas_gbq
# immunotherapy
import pandas
import os

# This query represents dataset "Immunotherapy" for domain "drug" and was generated for All of Us Registered Tier Dataset v8
dataset_64261871_drug_sql = """
    SELECT
        d_exposure.person_id,
        d_exposure.drug_concept_id,
        d_standard_concept.concept_name as standard_concept_name,
        d_standard_concept.concept_code as standard_concept_code,
        d_standard_concept.vocabulary_id as standard_vocabulary,
        d_exposure.drug_exposure_start_datetime,
        d_type.concept_name as drug_type_concept_name,
        d_visit.concept_name as visit_occurrence_concept_name,
        d_exposure.drug_source_concept_id,
        d_source_concept.concept_name as source_concept_name,
        d_source_concept.concept_code as source_concept_code,
        d_source_concept.vocabulary_id as source_vocabulary,
        
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.drug_exposure` d_exposure 
        WHERE
            (
                drug_concept_id IN (SELECT
                    DISTINCT ca.descendant_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                JOIN
                    (SELECT
                        DISTINCT c.concept_id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id             
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                        WHERE
                            concept_id IN (1314273, 1387104, 1397141, 1593273, 1594034, 35200783, 40238188, 42629079, 44818489, 45775965, 45892628)             
                            AND full_text LIKE '%_rank1]%'       ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) b 
                        ON (ca.ancestor_id = b.concept_id)))  
                    AND (d_exposure.PERSON_ID IN (SELECT
                        distinct person_id  
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_ehr_data = 1 ) )
            )) d_exposure 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_standard_concept 
            ON d_exposure.drug_concept_id = d_standard_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_type 
            ON d_exposure.drug_type_concept_id = d_type.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_route 
            ON d_exposure.route_concept_id = d_route.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
            ON d_exposure.visit_occurrence_id = v.visit_occurrence_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_visit 
            ON v.visit_concept_id = d_visit.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_source_concept 
            ON d_exposure.drug_source_concept_id = d_source_concept.concept_id"""

dataset_64261871_drug_df = pandas_gbq.read_gbq(
    dataset_64261871_drug_sql,
    dialect="standard",
    project_id=BIGQUERY_PROJECT,
    location=BIGQUERY_LOCATION,
    use_bqstorage_api=USE_BQSTORAGE_API,
    progress_bar_type="tqdm_notebook")

## Combine death sources

In [ ]:
import pandas_gbq
import pandas as pd
dataset = os.environ["WORKSPACE_CDR"]
all_death_data= pandas_gbq.read_gbq(f'''
WITH combined_death_data as 
(SELECT distinct person_id, death_date, death_type_concept_id, cause_concept_id, cast(primary_death_record as string) as primary_death_record
FROM `{dataset}.aou_death`
UNION DISTINCT
SELECT distinct person_id, death_date, death_type_concept_id, cause_concept_id, 'n/a' as primary_death_record
FROM `{dataset}.death`
)
SELECT person_id, death_date, t.concept_name as death_type, c.concept_name as death_cause, primary_death_record
from combined_death_data
LEFT JOIN `{dataset}.concept` t on t.concept_id = death_type_concept_id
LEFT JOIN `{dataset}.concept` c on c.concept_id = cause_concept_id ''',
    project_id=BIGQUERY_PROJECT, location=BIGQUERY_LOCATION,
    use_bqstorage_api=USE_BQSTORAGE_API, dialect='standard')

## Define the analysis population

The population includes patients with a diagnosis in `CANCER_CODES` whose earliest observed chemotherapy or immunotherapy exposure is on or after the first qualifying cancer diagnosis. Same-day initiation of both arms is excluded. `eligible_cohort` contains one row per selected patient, and `eligible_person_ids` drives the export.

In [ ]:
cancer_condition_dates = dataset_33674864_condition_df.copy()
cancer_condition_dates["condition_start_datetime"] = pandas.to_datetime(
    cancer_condition_dates["condition_start_datetime"],
    utc=True,
    errors="coerce",
)
cancer_condition_dates["icd_level3"] = (
    cancer_condition_dates["condition_source_value"]
    .fillna(cancer_condition_dates["standard_concept_code"])
    .astype(str)
    .str.split(".")
    .str[0]
)
cancer_condition_dates = cancer_condition_dates.dropna(subset=["condition_start_datetime"])
cancer_condition_dates = cancer_condition_dates.loc[
    cancer_condition_dates["icd_level3"].isin(CANCER_CODES)
].copy()

chemotherapy_exposures = dataset_08473385_drug_df.copy()
chemotherapy_exposures["drug_exposure_start_datetime"] = pandas.to_datetime(
    chemotherapy_exposures["drug_exposure_start_datetime"],
    utc=True,
    errors="coerce",
)
chemotherapy_exposures["study_drug_class"] = "chemotherapy"
chemotherapy_exposures = chemotherapy_exposures.dropna(subset=["drug_exposure_start_datetime"])

immunotherapy_exposures = dataset_64261871_drug_df.copy()
immunotherapy_exposures["drug_exposure_start_datetime"] = pandas.to_datetime(
    immunotherapy_exposures["drug_exposure_start_datetime"],
    utc=True,
    errors="coerce",
)
immunotherapy_exposures["study_drug_class"] = "immunotherapy"
immunotherapy_exposures = immunotherapy_exposures.dropna(subset=["drug_exposure_start_datetime"])

study_exposures = pandas.concat(
    [chemotherapy_exposures, immunotherapy_exposures],
    ignore_index=True,
)

death_dates = all_death_data.copy()
death_dates["death_date"] = pandas.to_datetime(
    death_dates["death_date"],
    utc=True,
    errors="coerce",
)
death_dates = death_dates.dropna(subset=["death_date"])


def first_person_date(frame, date_col):
    return frame.groupby("person_id")[date_col].min()


def last_person_date(frame, date_col):
    return frame.groupby("person_id")[date_col].max()


first_condition_date = first_person_date(cancer_condition_dates, "condition_start_datetime")
last_condition_date = last_person_date(cancer_condition_dates, "condition_start_datetime")
first_cancer_date = first_condition_date.copy()
first_death_date = first_person_date(death_dates, "death_date")

first_chemo_date = first_person_date(chemotherapy_exposures, "drug_exposure_start_datetime")
first_immunotherapy_date = first_person_date(immunotherapy_exposures, "drug_exposure_start_datetime")
last_chemo_date = last_person_date(chemotherapy_exposures, "drug_exposure_start_datetime")
last_immunotherapy_date = last_person_date(immunotherapy_exposures, "drug_exposure_start_datetime")
last_study_treatment_date = last_person_date(study_exposures, "drug_exposure_start_datetime")

last_record_date = pandas.concat(
    [
        last_condition_date.rename("last_condition_date"),
        last_chemo_date.rename("last_chemo_date"),
        last_immunotherapy_date.rename("last_immunotherapy_date"),
        first_death_date.rename("death_date"),
    ],
    axis=1,
).max(axis=1)

patient_ids = pandas.Index(
    pandas.concat(
        [
            cancer_condition_dates["person_id"],
            chemotherapy_exposures["person_id"],
            immunotherapy_exposures["person_id"],
        ],
        ignore_index=True,
    ).dropna().unique(),
    name="person_id",
)

patient_timeline = pandas.DataFrame(index=patient_ids)
patient_timeline = patient_timeline.join(first_condition_date.rename("first_condition_date"))
patient_timeline = patient_timeline.join(last_condition_date.rename("last_condition_date"))
patient_timeline = patient_timeline.join(first_cancer_date.rename("first_cancer_date"))
patient_timeline = patient_timeline.join(first_death_date.rename("death_date"))
patient_timeline = patient_timeline.join(first_chemo_date.rename("first_chemo_date"))
patient_timeline = patient_timeline.join(first_immunotherapy_date.rename("first_immunotherapy_date"))
patient_timeline = patient_timeline.join(last_chemo_date.rename("last_chemo_date"))
patient_timeline = patient_timeline.join(last_immunotherapy_date.rename("last_immunotherapy_date"))
patient_timeline = patient_timeline.join(last_study_treatment_date.rename("last_study_treatment_date"))
patient_timeline = patient_timeline.join(last_record_date.rename("last_record_date"))

patient_timeline["index_treatment_date"] = patient_timeline[
    ["first_chemo_date", "first_immunotherapy_date"]
].min(axis=1)
patient_timeline["treatment_group"] = "untreated"
patient_timeline["index_treatment_class"] = "none"

same_day_cross_arm_start = (
    patient_timeline["first_chemo_date"].notna()
    & patient_timeline["first_immunotherapy_date"].notna()
    & (patient_timeline["first_chemo_date"] == patient_timeline["first_immunotherapy_date"])
)
chemo_first = patient_timeline["first_chemo_date"].notna() & (
    patient_timeline["first_immunotherapy_date"].isna()
    | (patient_timeline["first_chemo_date"] < patient_timeline["first_immunotherapy_date"])
)
immunotherapy_first = patient_timeline["first_immunotherapy_date"].notna() & (
    patient_timeline["first_chemo_date"].isna()
    | (patient_timeline["first_immunotherapy_date"] < patient_timeline["first_chemo_date"])
)

patient_timeline.loc[chemo_first, "treatment_group"] = "chemotherapy_initiator"
patient_timeline.loc[immunotherapy_first, "treatment_group"] = "immunotherapy_initiator"
patient_timeline.loc[same_day_cross_arm_start, "treatment_group"] = "same_day_cross_arm_start"
patient_timeline.loc[chemo_first, "index_treatment_class"] = "chemotherapy"
patient_timeline.loc[immunotherapy_first, "index_treatment_class"] = "immunotherapy"
patient_timeline.loc[same_day_cross_arm_start, "index_treatment_class"] = "same_day_cross_arm_start"

later_chemo = patient_timeline["first_chemo_date"] > patient_timeline["index_treatment_date"]
later_immunotherapy = (
    patient_timeline["first_immunotherapy_date"] > patient_timeline["index_treatment_date"]
)

patient_timeline["switches_treatment_class"] = False
patient_timeline.loc[
    patient_timeline["treatment_group"] == "chemotherapy_initiator",
    "switches_treatment_class",
] = later_immunotherapy
patient_timeline.loc[
    patient_timeline["treatment_group"] == "immunotherapy_initiator",
    "switches_treatment_class",
] = later_chemo

patient_timeline["days_from_cancer_to_treatment"] = (
    patient_timeline["index_treatment_date"] - patient_timeline["first_cancer_date"]
).dt.days
patient_timeline["condition_baseline_days"] = (
    patient_timeline["index_treatment_date"] - patient_timeline["first_condition_date"]
).dt.days
patient_timeline["condition_followup_days"] = (
    patient_timeline["last_condition_date"] - patient_timeline["index_treatment_date"]
).dt.days
patient_timeline["overall_followup_days_after_index"] = (
    patient_timeline["last_record_date"] - patient_timeline["index_treatment_date"]
).dt.days
patient_timeline["followup_days_after_index"] = patient_timeline["condition_followup_days"]
patient_timeline["days_from_index_to_last_treatment"] = (
    patient_timeline["last_study_treatment_date"] - patient_timeline["index_treatment_date"]
).dt.days
patient_timeline["death_after_index"] = (
    patient_timeline["death_date"] > patient_timeline["index_treatment_date"]
)
patient_timeline["days_from_index_to_death"] = (
    patient_timeline["death_date"] - patient_timeline["index_treatment_date"]
).dt.days

patient_timeline["has_cancer"] = patient_timeline["first_cancer_date"].notna()
patient_timeline["has_clear_treatment_arm"] = patient_timeline["treatment_group"].isin(
    ["chemotherapy_initiator", "immunotherapy_initiator"]
)
patient_timeline["treatment_after_cancer"] = (
    patient_timeline["days_from_cancer_to_treatment"] >= 0
)
patient_timeline["trajectory_candidate"] = (
    patient_timeline["has_cancer"]
    & patient_timeline["has_clear_treatment_arm"]
    & patient_timeline["treatment_after_cancer"]
)
patient_timeline["baseline_window_ok"] = (
    patient_timeline["condition_baseline_days"] >= BASELINE_DAYS_MIN
)
patient_timeline["followup_window_ok"] = (
    patient_timeline["condition_followup_days"] >= FOLLOWUP_DAYS_MIN
)
patient_timeline["base_candidate"] = (
    patient_timeline["trajectory_candidate"]
    & patient_timeline["baseline_window_ok"]
    & patient_timeline["followup_window_ok"]
)

for outcome_name, window_days in MORTALITY_WINDOWS_DAYS.items():
    window_slug = outcome_name.lower().replace(" ", "_")
    patient_timeline[f"{window_slug}_observed"] = (
        patient_timeline["overall_followup_days_after_index"] >= window_days
    ) | (
        patient_timeline["death_after_index"]
        & (patient_timeline["days_from_index_to_death"] <= window_days)
    )
    patient_timeline[f"{window_slug}_event"] = (
        patient_timeline["death_after_index"]
        & (patient_timeline["days_from_index_to_death"] <= window_days)
    )
    patient_timeline[f"{window_slug}_candidate"] = (
        patient_timeline["base_candidate"]
        & patient_timeline[f"{window_slug}_observed"]
    )

eligible_cohort = patient_timeline.loc[patient_timeline["trajectory_candidate"]].reset_index().copy()
eligible_person_ids = eligible_cohort["person_id"].unique()

## Cohort diagnostics

The tables report attrition, condition-coverage summaries, mortality over the configured 24- and 36-month windows, and post-index crossover. Mortality-window summaries require sufficient observed follow-up or a death within that window. These are descriptive checks; the downstream experiment defines its own outcome window.

In [ ]:
attrition_masks = {
    "Cancer diagnosis in provided code list": patient_timeline["has_cancer"],
    "Clear chemotherapy vs immunotherapy arm": (
        patient_timeline["has_cancer"] & patient_timeline["has_clear_treatment_arm"]
    ),
    "Treatment after cancer diagnosis": patient_timeline["trajectory_candidate"],
    f"baseline >= {BASELINE_DAYS_MIN} days": (
        patient_timeline["trajectory_candidate"] & patient_timeline["baseline_window_ok"]
    ),
    f"follow-up >= {FOLLOWUP_DAYS_MIN} days": patient_timeline["base_candidate"],
}

attrition_df = pandas.DataFrame(
    {
        "step": list(attrition_masks.keys()),
        "patients": [int(mask.sum()) for mask in attrition_masks.values()],
    }
)

arm_summary = (
    patient_timeline.loc[patient_timeline["base_candidate"]]
    .groupby("treatment_group")
    .agg(
        patients=("treatment_group", "size"),
        median_days_from_cancer_to_treatment=("days_from_cancer_to_treatment", "median"),
        median_condition_baseline_days=("condition_baseline_days", "median"),
        median_condition_followup_days=("condition_followup_days", "median"),
        median_overall_followup_days=("overall_followup_days_after_index", "median"),
        switchers=("switches_treatment_class", "sum"),
    )
    .reset_index()
)

outcome_summary_frames = []
outcome_timing_frames = []

for outcome_name, window_days in MORTALITY_WINDOWS_DAYS.items():
    window_slug = outcome_name.lower().replace(" ", "_")
    candidate_mask = patient_timeline[f"{window_slug}_candidate"]
    event_mask = patient_timeline[f"{window_slug}_event"]

    outcome_summary_frames.append(
        patient_timeline.loc[candidate_mask]
        .groupby("treatment_group")
        .agg(
            patients=("treatment_group", "size"),
            events=(f"{window_slug}_event", "sum"),
            event_rate=(f"{window_slug}_event", "mean"),
            median_days_from_cancer_to_treatment=("days_from_cancer_to_treatment", "median"),
            median_condition_baseline_days=("condition_baseline_days", "median"),
            median_condition_followup_days=("condition_followup_days", "median"),
            median_overall_followup_days=("overall_followup_days_after_index", "median"),
            switchers=("switches_treatment_class", "sum"),
        )
        .assign(outcome=outcome_name, window_days=window_days)
        .reset_index()
    )

    event_frame = patient_timeline.loc[candidate_mask & event_mask]
    if event_frame.empty:
        outcome_timing_frames.append(
            pandas.DataFrame(
                columns=[
                    "treatment_group",
                    "events",
                    "median_days_from_index",
                    "outcome",
                    "window_days",
                ]
            )
        )
    else:
        outcome_timing_frames.append(
            event_frame.groupby("treatment_group")
            .agg(
                events=("treatment_group", "size"),
                median_days_from_index=("days_from_index_to_death", "median"),
            )
            .assign(outcome=outcome_name, window_days=window_days)
            .reset_index()
        )

outcome_summary = pandas.concat(outcome_summary_frames, ignore_index=True)
outcome_timing_summary = pandas.concat(outcome_timing_frames, ignore_index=True)

crossover_summary = (
    eligible_cohort.groupby("treatment_group")
    .agg(patients=("person_id", "size"), switchers=("switches_treatment_class", "sum"))
    .reset_index()
)
crossover_summary["switcher_share"] = crossover_summary["switchers"] / crossover_summary["patients"]

display(attrition_df)
display(arm_summary)
display(outcome_summary)
display(outcome_timing_summary)
display(crossover_summary)

## Export events and death tokens

In [ ]:
events_df = pd.read_pickle((SHARED_DATA_DIR / 'events.pkl'))

In [ ]:
icd_conditions_candidates = events_df.loc[
    events_df["person_id"].isin(eligible_person_ids)
].copy()

death_events = all_death_data[["person_id", "death_date"]].copy()
death_events["timestamp"] = pd.to_datetime(
    death_events["death_date"],
    utc=True,
    errors="coerce",
)
death_events = death_events.dropna(subset=["timestamp"])
death_events = death_events.loc[
    death_events["person_id"].isin(eligible_person_ids)
].copy()
death_events = death_events.sort_values(["person_id", "timestamp"])
death_events = death_events.drop_duplicates(subset=["person_id"], keep="first")
death_events = death_events.assign(token_col="DEATH", type="death")
death_events = death_events[["person_id", "timestamp", "token_col", "type"]]

icd_conditions_candidates = pd.concat(
    [icd_conditions_candidates, death_events],
    ignore_index=True,
)
icd_conditions_candidates = icd_conditions_candidates.sort_values(
    ["person_id", "timestamp", "type", "token_col"]
).reset_index(drop=True)

print(f"Candidates: {len(eligible_person_ids)}")
print(f"Death rows appended: {len(death_events)}")

In [ ]:
dataset_08473385_drug_df.to_pickle(OUTPUT_DIR / "chemotherapy_df.pkl")
dataset_64261871_drug_df.to_pickle(OUTPUT_DIR / "immunotherapy_df.pkl")
icd_conditions_candidates.to_pickle(OUTPUT_DIR / "events.pkl")